### Mouse brain - cell2location paper, Figure 2 (mb01)

**Paper**: Kleshchevnikov V. *et al.* *Comprehensive mapping of tissue cell architecture via integrated single cell and spatial transcriptomics.* bioRxiv 2020.11.15.378125 (published as *Cell2location maps fine-grained cell types in spatial transcriptomics*, Nat Biotechnol 40, 661-671, 2022).


The authores: "We generated matched single nucleus (sn) and Visium spatial RNA-seq (10X Genomics) profiles of adjacent mouse brain sections that contain multiple regions from the telencephalon and diencephalon (Fig 2A). 

To assess the biological and intra-organ technical variation in spatial mapping, we assayed two mouse brains and serial tissue sections from each brain (total of 3 and 2 matched sections from two animals, respectively, and an extra section for snRNA-seq), creating a rich multi-modal and replicated transcriptomic dataset."


![Fig 2A](../mouse_brain/figures/fig_2A.png)


This notebook reproduces the mouse brain panels of **Fig 2** 

#### Libs: **src/libs**

- **single_cell_lib** methods:
  - Class: SingleCell
    - generic Scanpy
    - Squidpy 
    - cell2location - 
    - LIANA 
    - **DEFAULT_PARAMS**

- **single_cell_neuro_lib** - mouse brain methods and **NEURO_PARAMS**
  - Classe: SingleCellNeuro(SingleCell)

<br>

| section | content | paper |
|---|---|---|
| 1 | open the h5ad files | |
| 2 | components and attributes | |
| 3 | filter data | Methods: genes < 5000, mito < 10%, doublets < 0.2 |
| 4 | quality control | Fig S4 |
| 5 | UMAP: broad classes and 59 subtypes | Fig 2B |
| 6 | anatomical regions, major regional subtypes, sparse cell types, fine anatomical subtypes | Fig 2C-F |
| 7 | Slide-seq V2 mapping | Fig 2H |
| 8 | quantitative validation workflow (manually dissected layers) | Fig 2I |
| 9 | heatmap of relative cell subtype proportion per layer | Fig 2J |

**Data** (**mouse_brain/data**): 5 Visium sections (E-MTAB-11114) and snRNA-seq (E-MTAB-11115) from the cell2location server; the paper's own cell2location map with region clusters (sections ST8059048, mouse 1 and ST8059051, mouse 2); squidpy Slide-seqV2; Allen mouse cortex + hippocampus Smart-seq (Yao et al. 2020).

<div class="alert alert-warning">
<b>Differences from the paper</b>
</div>

- **Fig 2C-F** use the paper's precomputed cell2location map (2 of the 5 sections); region names come from cell-type enrichment rules (**region_celltype_map**), not from manual annotation.

![fig 2C-F](../mouse_brain/figures/fig2C-F.png)


- **Fig 2H** uses squidpy's Slide-seqV2 hippocampus puck (41,786 beads, 4,000 HVGs, counts recovered from log-normalised X), not the paper's 53,208-bead section; mapping with current cell2location (v0.1.5) priors.

![fig 2H](../mouse_brain/figures/Fig2H.png)


- **Fig 2I/J**: the manual Loupe layer annotation of SSp is not published; cortical layers are taken from the region clusters (all cortex, not SSp only). The Allen taxonomy release gives 104 + 34 cell types (paper: 98 + 23).

![fig 2I-J](../mouse_brain/figures/Fig2I-Jb.png)


- cell2location runs in the sandbox were done on CPU with reduced epochs (tags **cpu3000**, **cpu2000**); rerun with **src/programs/run_c2l_mouse_brain.py --max_epochs 30000 --accelerator gpu** and change the tags below.



## Loading packages

In [ ]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_lib import cmap_lbwr          # light blue -> white -> light red (dot plots)
from libs.single_cell_neuro_lib import SingleCellNeuro, NEURO_PARAMS
from libs.matplotlib_figure_style import apply_figure_style
from libs.Basic import create_dir, pdwritecsv, pdreadcsv

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
print({p: version(p) for p in ['scanpy', 'squidpy', 'anndata', 'cell2location', 'liana']})

## Parameters

All file names, cutoffs and model settings are in the class (**src/libs/single_cell_neuro_lib.py**): **NEURO_PARAMS** (over **DEFAULT_PARAMS**) and **NEURO_FILES['inputs']** + **['mb01']** (**scn.fname(key)** = the name for the methods, **scn.file_path(key)** = the full path). Override with `SingleCellNeuro(ROOT0, analysis='mb01', params=dict(...))`; **scn.calc_df_params()** lists every value.

In [ ]:
force=False
verbose=True

apply_figure_style()
# parameters: DEFAULT_PARAMS < NEURO_PARAMS (QC cutoffs snrna_*, density_quantile, region_label_override, library_main,
#             cell2location caches c2l_tag_*, c2l_train_if_missing, c2l_max_epochs)    (params=dict(...) overrides)
# files:      NEURO_FILES['inputs'] + NEURO_FILES['mb01']                            (scn.fname(key) / scn.file_path(key))
scn = SingleCellNeuro(ROOT0, analysis='mb01')
root_data, root_figure, root_result = scn.root_data, scn.root_figure, scn.root_result
print(scn)
print(root_data, '\n', sorted(os.listdir(root_data)))

In [ ]:
# QC parameters (no doublet filter: the scrublet column is absent in the public file)
{k: scn.params[k] for k in ['snrna_max_genes', 'snrna_max_mt_frac', 'density_quantile', 'region_label_override', 'accelerator']}

In [ ]:
df_params = scn.calc_df_params()
df_params[df_params.value.astype(str) != df_params.default.astype(str)]

### 1. Open the h5ad files

- **adata_vis**: 
  - 5 Visium sections merged (**download_mouse_brain_visium.py**), 
  - Ensembl **var_names**, symbols in **var['SYMBOL']**

- **adata_ref**: 
  - snRNA-seq raw counts, 
  - 6 samples (5 paired with Visium + 1 extra section)

In [ ]:
adata_vis = scn.read_h5ad(scn.fname('visium'), set_self=False)
adata_ref = scn.read_h5ad(scn.fname('reference'), set_self=False)
adata_vis, adata_ref

### 2. Components and attributes

In [ ]:
scn.summary(adata_vis)

In [ ]:
df_comp_vis = scn.calc_df_components(adata_vis)
df_comp_vis

In [ ]:
df_comp_vis.groupby('slot').size().to_frame('n_keys').T

In [ ]:
adata_vis.obs.columns

In [ ]:
scn.sample_col

In [ ]:
df_sections = adata_vis.obs.groupby(scn.sample_col, observed=True).agg(
    n_spots=('in_tissue', 'size'), in_tissue=('in_tissue', 'sum'))
df_sections

In [ ]:
df_sections = adata_vis.obs.groupby(scn.sample_col, observed=True).agg(
    n_spots=('in_tissue', 'size'), in_tissue=('in_tissue', 'sum'))
df_sections['slide'] = [s for s in df_sections.index]
df_sections['image_hires'] = [adata_vis.uns['spatial'][s]['images']['hires'].shape for s in df_sections.index]
df_sections

In [ ]:
scn.summary(adata_ref)

In [ ]:
df_samples_ref = adata_ref.obs[scn.sample_col].value_counts().to_frame('n_nuclei')
df_samples_ref

### 3. Filter data

**snRNA-seq** (paper Methods) - kept:

  - nuclei with < 5000 detected genes, 
  - mitochondrial fraction < 0.10 and 
  - scrublet doublet score < 0.2. 
  
  - Visium spots contain more than one cell; therefore, no doublet-score corrections are needed.
  - The paper's annotation then defines the 40,532 nuclei used as reference.

  - For Visium you're right: a spot captures several cells, so doublet scores don't apply. 
  - For the snRNA-seq reference the paper did filter with scrublet < 0.2, but the 40,532 annotated nuclei are already the post-filtering set, so re-applying that cutoff adds nothing. I'll remove the doublet criterion and its warning, and state both reasons in the notebook.

In [ ]:
adata_ref, df_filter_ref = scn.filter_snrna(adata_ref, inplace=False)
print(f"kept {df_filter_ref.attrs['kept']:,} of {df_filter_ref.attrs['total']:,} nuclei")
df_filter_ref

In [ ]:
adata_ref, df_annot = scn.add_paper_annotation(scn.fname('reference_annotation'), adata_ref, verbose=verbose)
print(f"annotation: {len(df_annot):,} nuclei, {df_annot[scn.params['celltype_col']].nunique()} cell types; "
      f"after QC + annotation: {adata_ref.n_obs:,}")
adata_ref.obs[scn.params['broad_col']].value_counts().to_frame('n_nuclei')

### oligodendrocyte

An oligodendrocyte is a specialized type of glial cell in the central nervous system (CNS)—comprising the brain and spinal cord—whose primary function is to form the myelin sheath around neuronal axons. [1](https://en.wikipedia.org/wiki/Oligodendrocyte), [2](https://www.cancer.gov/publications/dictionaries/cancer-terms/def/oligodendrocyte)

Key Functions

- Myelination: They wrap layers of lipid-rich membrane around axons, providing electrical insulation. This allows for saltatory conduction, a process where electrical impulses jump rapidly from one gap (node of Ranvier) to the next, speeding up signal transmission. [1](https://www.kenhub.com/en/library/physiology/oligodendrocytes), [2](https://www.khanacademy.org/test-prep/mcat/organ-systems/neural-cells/v/oligodendrocytes), [3](https://www.youtube.com/watch?v=lNMibSvlj3k), [4](https://parisbraininstitute.org/glossary/oligodendrocyte), [5](https://www.assaygenie.com/blog/what-are-oligodendrocytes-functions-markers-disease-links/)
- Multiple Axon Coverage: A single oligodendrocyte can extend its branches to myelinate up to 40 to 50 separate axon segments. (In contrast, Schwann cells perform myelination in the peripheral nervous system, with one cell covering just a single axon segment). [1](https://en.wikipedia.org/wiki/Oligodendrocyte), [2](https://www.assaygenie.com/blog/what-are-oligodendrocytes-functions-markers-disease-links/), [3](https://www.youtube.com/watch?v=lNMibSvlj3k)
- Metabolic Support: Beyond insulation, they deliver vital metabolic nutrients (such as lactate and pyruvate) to support long-term axon health and survival. [1](https://www.biolegend.com/ja-jp/oligodendrocytes), [2](https://www.assaygenie.com/blog/what-are-oligodendrocytes-functions-markers-disease-links/)

#### Development & Lineage

Oligodendrocytes develop from oligodendrocyte progenitor cells (OPCs) that originate in the embryonic neural tube. The lineage progresses from OPCs to pre-myelinating/immature oligodendrocytes before maturing into functional myelin-forming cells. [1](https://www.sciencedirect.com/topics/neuroscience/oligodendrocyte), [2](https://en.wikipedia.org/wiki/Oligodendrocyte), [3](https://www.kenhub.com/en/library/physiology/oligodendrocytes)

#### Clinical Significance

Damage to oligodendrocytes or the myelin they produce leads to demyelination, which disrupts nerve signal flow. This is a central feature in autoimmune conditions like multiple sclerosis (MS), as well as genetic leukodystrophies (such as Pelizaeus-Merzbacher disease) and brain tumors like oligodendroglioma. [1](https://www.youtube.com/watch?v=lNMibSvlj3k), [2](https://www.kenhub.com/en/library/physiology/oligodendrocytes), [3](https://www.biolegend.com/ja-jp/oligodendrocytes)
Search on National Cancer Institute Dictionary or explore cellular details via the [Paris Brain Institute Glossary](https://parisbraininstitute.org/glossary/oligodendrocyte).


### oligodendrocyte precursor cell (OPC)

An oligodendrocyte precursor cell (OPC)—also known as NG2-glia or polydendrocyte—is a specialized type of glial cell in the central nervous system (CNS) that acts as the primary progenitor for oligodendrocytes (the cells that form myelin sheaths). [1](https://en.wikipedia.org/wiki/Oligodendrocyte_progenitor_cell)

#### Key Characteristics

- Identity & Markers: Identified in humans and rodents by co-expression of surface markers such as PDGFRA (platelet-derived growth factor receptor alpha) and CSPG4 (NG2). [1](https://www.biocompare.com/Editorial-Articles/590587-A-Guide-to-Oligodendrocyte-Markers/), [2](https://en.wikipedia.org/wiki/Oligodendrocyte_progenitor_cell)
- Distribution: They are widely and evenly distributed across both grey and white matter in the brain and spinal cord, making up about 3% to 5% of all CNS cells. [1](https://www.nature.com/articles/s41593-023-01423-3), [2](https://en.wikipedia.org/wiki/Oligodendrocyte_progenitor_cell)
- Persistence: Unlike many progenitor cells that disappear after early development, OPCs persist throughout adulthood and retain the ability to migrate and proliferate. [1](https://www.pnas.org/doi/10.1073/pnas.2202580119), [2](https://pmc.ncbi.nlm.nih.gov/articles/PMC10910408)

#### Core Functions

- Myelinogenesis and Remyelination: Their canonical role is to differentiate into mature oligodendrocytes during development and actively repair damaged myelin (remyelination) in pathological states like multiple sclerosis. [1](https://pubmed.ncbi.nlm.nih.gov/31614602/), [2](https://www.sciencedirect.com/topics/neuroscience/oligodendrocyte-progenitor), [3](https://en.wikipedia.org/wiki/Oligodendrocyte_progenitor_cell)
- Synaptic Integration and Communication: OPCs receive direct synaptic input from neurons (forming functional chemical synapses via AMPA receptors) and can modulate neuronal activity. [1](https://www.youtube.com/watch?v=iJmH8G9pces&t=368), [2](https://www.nature.com/articles/s41593-023-01423-3)
- Circuit Remodeling and Phagocytosis: Beyond making myelin, OPCs contribute to non-myelinating roles, such as helping sculpt neural circuits, regulating blood vessel growth (angiogenesis), and clearing/engulfing excess synapses or axonal debris. [1](https://www.nature.com/articles/s41467-025-66521-1), [2](https://www.nature.com/articles/s41593-023-01423-3), [3](https://www.pnas.org/doi/10.1073/pnas.2202580119)
If you'd like to explore further, let me know:
- Are you interested in their role in a specific disease (like multiple sclerosis or gliomas)?
- Do you want to know more about OPC differentiation markers or lab culture protocols?


#### regression_signatures

**fname_signatures** = 'snrna/regression_signatures_sc.h5ad' - is the paper's own trained reference model, downloaded from the cell2location data server.

- **Source:** https://cell2location.cog.sanger.ac.uk/tutorial/mouse_brain_snrna/regression_model/RegressionGeneBackgroundCoverageTorch_65covariates_40532cells_12819genes/sc.h5ad

- **Local name:** renamed **sc.h5ad** to **regression_signatures_sc.h5ad** during the download step, so it wouldn't be confused with other **sc.h5ad** files.

- **What it is:** the negative-binomial regression model from the original paper, made with cell2location v0.05 (model class **RegressionGeneBackgroundCoverageTorch**). It was fitted on the 40,532 annotated snRNA-seq nuclei and 12,819 genes, as the folder name says. This is the paper's "Expression signatures of 59 mouse brain reference cell types … derived using a negative binomial regression model" (Methods, p. 21).
- **Inside the file:** **uns['regression_mod']['post_sample_means']['gene_factors']** holds a 65 × 12,819 matrix, one row per model factor. **scn.signatures_from_regression()** keeps the 59 rows whose names match the **annotation_1** cell types. The other 6 are further covariates of the model, most likely the 6 snRNA-seq samples; I didn't use them. It converts Ensembl IDs to symbols via **var['SYMBOL']** and returns a genes × 59 table.

Where it is used:
- **Section 3:** counting the Visium genes shared with the reference (12,809, as in the paper).
- **Fig 2H:** the signatures mapped onto the Slide-seq beads (2,528 shared genes).
- ****mouse_brain_cell_types.tsv**:** the **top_signature_genes** column.

It is not used for Fig 2C–F, which read the paper's precomputed cell2location map, nor for Fig 2J, which uses the Allen Smart-seq per-cluster averages.

The same server folder has the model's other outputs (**covariate_effects.csv**, **sample_effects.csv**, **model_.p**), which I didn't download.

In [ ]:
print(scn.fname('signatures'))
df_sig_ref = scn.signatures_from_regression(scn.fname('signatures'), use_symbols=False)
df_sig_ref.head(3).T.head(8)

#### Visium 

  - spots under tissue only; 
  - mitochondrial genes removed for mapping (their counts kept in **obsm['MT']**);
  - genes shared with the reference signatures (paper: 12,809 genes, 14,968 locations).

In [ ]:
adata_vis = adata_vis[adata_vis.obs['in_tissue'] == 1].copy()

n_genes0 = adata_vis.n_vars

adata_vis = scn.remove_mt_genes(adata_vis)
n_shared = len(np.intersect1d(adata_vis.var_names, df_sig_ref.index))

print(f"Visium: {adata_vis.n_obs:,} spots; genes {n_genes0:,} -> {adata_vis.n_vars:,} without MT; "
      f"shared with the 59 signatures: {n_shared:,}")

### 4. Quality control

#### adata_vis

In [ ]:
scn.calc_qc_metrics(adata_vis)

# MT genes were removed in section 3; their counts are in obsm['MT'] -> mito % of all counts
mt_counts = np.asarray(adata_vis.obsm['MT']).sum(1)

adata_vis.obs['pct_counts_mt'] = 100 * mt_counts / (adata_vis.obs['total_counts'].values + mt_counts)

fig = scn.plot_qc(adata=adata_vis, figsize=(20, 5), fontsize_scale=1.6, fname=scn.fname('fig_qc_visium'), title='Visium spots per section', force=force, verbose=verbose)
plt.show()

In [ ]:
df_qc_vis = scn.calc_df_qc(adata=adata_vis)
df_qc_vis.T

#### adata_ref

In [ ]:
fig = scn.plot_qc(adata=adata_ref, figsize=(20, 5), fontsize_scale=1.6, fname=scn.fname('fig_qc_snrna'), title='snRNA-seq nuclei per sample (after QC)',
                 thresholds={'n_genes_by_counts': scn.params['snrna_max_genes'],
                             'pct_counts_mt': 100 * scn.params['snrna_max_mt_frac']}, force=force, verbose=verbose)
plt.show()

In [ ]:
df_qc_ref = scn.calc_df_qc(adata=adata_ref)
df_qc_ref.T

In [ ]:
df_ct_sample = pd.crosstab(adata_ref.obs[scn.params['celltype_col']], adata_ref.obs[scn.sample_col])
print("cell types present in all 6 samples:", int((df_ct_sample > 0).all(1).sum()), "of", len(df_ct_sample))
df_ct_sample.head(10)

### 5. UMAP - broad classes and 59 subtypes (Fig 2B)

The UMAP coordinates, **annotation_1** and **broad_class** are the paper's (cellxgene object); only obs and obsm are read.

In [ ]:
scn.fname('umap')

In [ ]:
adata_umap = scn.load_umap_obs(scn.fname('umap'))
fig = scn.plot_umap_broad_subtypes(adata_umap, figsize=(18, 8), fontsize_scale=1.8, fname=scn.fname('fig_2B'), force=force, verbose=verbose)
plt.show()
adata_umap.obs[scn.params['broad_col']].value_counts().to_frame('n_nuclei').T

### 6. Anatomical regions, major regional subtypes, sparse cell types, fine anatomical subtypes (Fig 2C-F)

The paper's cell2location map gives cells per spot (**q05_spot_factors<type>**, 59 types) and **region_cluster** (Leiden on abundance; paper Methods: kNN 38, resolution 1.3).

Each region cluster is named by the cell types most enriched in it (cluster mean / slide mean), using **region_celltype_map**.

### Abundance of cell types in regions

In [ ]:
adata_map = scn.load_paper_map(scn.fname('c2l_map'))
df_abund = scn.calc_df_abundance(adata_map)
print(adata_map.shape, scn.library_ids(adata_map), df_abund.shape)
df_abund.head(3).T.head(10)

In [ ]:
df_regions = scn.calc_df_regions(adata_map)
df_regions

**Edge check.** Two clusters are not labelled well by enrichment alone. A Visium spot has 6 neighbours; spots on the tissue border have fewer. Cluster 11 (Inh_Lamp5, Astro enriched) lies on the pial surface: it is **cortical layer 1**. Cluster 21 (LowQ_1) is a tissue-rim artefact. Both are set with **region_label_override**.

In [ ]:
df_edge = scn.calc_df_edge_fraction(adata_map)
df_regions_edge = df_regions.set_index('region_cluster').join(df_edge)
df_regions_edge.loc[df_regions_edge.mean_frac_edge > 0.25, ['label', 'top_types', 'mean_frac_edge']]

### Label with region_label

In [ ]:
scn.add_region_labels(df_regions, adata_map)
df_regions['label_final'] = df_regions.region_cluster.map(lambda c: scn.params['region_label_override'].get(c))
df_regions['label_final'] = df_regions['label_final'].fillna(df_regions['label'])
pdwritecsv(df_regions.join(df_edge[['mean_frac_edge']], on='region_cluster'), scn.fname('tsv_regions_spots'), root_result, verbose=True)

adata_map.obs['region_label'].value_counts().to_frame('n_spots')

In [ ]:
regions = np.unique(adata_map.obs.region_label)
regions

### Brain regions

In [ ]:
fname = scn.fname('region_info')
dfr = pdreadcsv(fname, root_data, verbose=True)
dfr

[mb01_brain_regions](../mouse_brain/figures/mb01_brain_regions.png)

In [ ]:
force=False
verbose=True

df_brain_regions = pdreadcsv(scn.fname('region_info'), root_data)
regions = dict(zip(df_brain_regions.region_id, df_brain_regions.region_name))   # 'CTX_L4' -> 'Isocortex layer 4 (granular layer)'

fig = scn.plotly_spatial_visium(names=regions, adata=adata_map, key='region_label', library_id=scn.params['library_main'],
                                fname='fig_regions_html', include_plotlyjs=True,      # self-contained HTML, {lib} = the section
                                force=force, verbose=verbose)
fig.show()

### Cell types

In [ ]:
cell_types = adata_map.uns['mod']['factor_names']
cell_types

In [ ]:
fname = scn.fname('cell_types')
dfct = pdreadcsv(fname, root_data, verbose=True)
dfct

In [ ]:
scn.assign_cortical_layers(adata_map)

adata_map.obs.columns

In [ ]:
cols = ['in_tissue', 'array_row', 'array_col', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts',
         'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 
         'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'sample_name', 'sample_name2', 
         'image_name', 'slide', 'position', 'leiden', 'region_cluster', 'region_label', 'cortical_layer'],


cols=['array_col', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 
      'log1p_total_counts',  'sample_name', 'image_name', 'slide', 'position', 'leiden', 
       'region_cluster', 'region_label', 'cortical_layer']

adata_map.obs[cols].head(3).T

### All samples/libs

In [ ]:
scn.library_ids(adata_map)

In [ ]:
force=False
verbose=True

for lib in scn.library_ids(adata_map):
    fig = scn.plot_regions_he(adata_map, library_id=lib, figsize=(12, 7), fontsize_scale=1.6, fname=scn.fname('fig_2C', lib=lib), force=force, verbose=verbose)
    plt.show()

### Plot subtypes

In [ ]:
force=False
verbose=True

panels = {'major regional subtypes': scn.params['fig2D_types'],
          'sparse cell types': scn.params['fig2E_types'],
          'fine anatomical subtypes': scn.params['fig2F_types']}

for lib in scn.library_ids(adata_map):
    fig = scn.plot_density_panels(panels, adata_map, library_id=lib, df_abund=df_abund, panel_size=6.5, 
                                  fontsize_scale=1.6, fname=scn.fname('fig_2DEF', lib=lib), 
                                  force=force, verbose=verbose)
    plt.show()


Colour scale per cell type: 

- 0 to the 99.2% quantile of cells per spot. 

- **Ext_L25** (L2-5 IT neurons, peak in L4) is shown as **Ext_L25 (L4)**: 
  - the 59-type annotation has no **Ext_L4**, the label used in Fig 2F.

### 7. Slide-seq V2 (Fig 2H)

The 59 snRNA signatures (paper NB regression) are mapped to a Slide-seqV2 puck of the mouse hippocampus (Stickels et al. 2021). Paper priors for 10-µm beads: 1 cell and 1 cell type per location.

squidpy stores **X = log1p(normalize_total)** on 4,000 HVGs with the raw library size in **obs['total_counts']**; counts are recovered exactly as **expm1(X) * total_counts / target**.

In [ ]:
adata_ss, df_sig_ss, diag_ss = scn.prepare_slideseq_mapping(scn.fname('slideseq'), scn.fname('signatures'))
print(adata_ss.shape)
print(adata_ss)
diag_ss

In [ ]:
root_ss = create_dir(scn.file_path('c2l_slideseq_dir', tag=scn.params['c2l_tag_slideseq']))
root_ss

In [ ]:
if (root_ss / 'sp.h5ad').exists() or scn.params['c2l_train_if_missing']:
    adata_ss_map = scn.train_mapping(adata_ss, df_sig_ss, root_ss, N_cells_per_location=scn.params['slideseq_N'],
                                    detection_alpha=scn.params['slideseq_alpha'], max_epochs=scn.params['c2l_max_epochs'],
                                    A_factors_per_location=scn.params['slideseq_A'],
                                    B_groups_per_location=scn.params['slideseq_B'], verbose=verbose)
    print(adata_ss_map.shape, {k: v for k, v in adata_ss_map.uns['c2l_training'].items() if k != 'elbo_history'})
else:
    adata_ss_map = None
    print(f"no cached run in {root_ss}: run src/programs/run_c2l_mouse_brain.py --mode slideseq")

In [ ]:
force=False
verbose=True

if adata_ss_map is not None:
    elbo = np.asarray(adata_ss_map.uns['c2l_training']['elbo_history'])
    fig, ax = plt.subplots(figsize=(4, 2.5))
    ax.plot(np.arange(len(elbo))[10:], elbo[10:], lw=0.8)
    ax.set_xlabel('epoch'); ax.set_ylabel('ELBO loss'); ax.set_title('Slide-seq mapping: training history')
    scn.style_figure(fig, scn.params['font_color'], scn.params['bg_color'], scn.params['fontsize_scale'], relayout=True)
    plt.show()
    fig = scn.plot_slideseq(adata_ss_map, panel_size=7.5, fontsize_scale=1.6, fname=scn.fname('fig_2H'), force=force, verbose=verbose)
    plt.show()

### 8. Quantitative validation workflow based on scRNA-seq from manually dissected layers (Fig 2I)

Allen Smart-seq cells from the primary somatosensory cortex (SSp) were dissected by layer (Yao et al. 2020). Following the paper:

- cell types of SSp plus types exclusive to the hippocampal subiculum (PAR-POST-PRE), with ≥ 3 cells

- signatures = per-cluster average expression (NB regression is not suited to Smart-seq)
- mapped with cell2location to the Visium cortical-layer spots (here: region clusters CTX_L1 ... CTX_L6b)

In [ ]:
adata_allen = scn.read_h5ad(scn.fname('allen'), set_self=False)
scn.sanitize_allen_types(adata_allen)          # '166_L2/3 IT CTX' -> '166_L2-3 IT CTX' ('/' is not allowed in h5ad keys)

df_meta_allen = adata_allen.obs

df_type_counts, types_ssp, types_hpc = scn.allen_select_types(df_meta_allen)

print(f"Allen cells: {adata_allen.n_obs:,} ({df_meta_allen[scn.params['allen_region_col']].value_counts().to_dict()}); "
      f"types: SSp {len(types_ssp)} + hippocampus-exclusive {len(types_hpc)} = {len(types_ssp) + len(types_hpc)}")

df_allen_counts, df_allen_prop = scn.calc_df_allen_layer_prop(df_meta_allen, types_ssp)
df_allen_counts.sum().to_frame('cells per layer (SSp)')

In [ ]:
force=False
verbose=True

scn.assign_cortical_layers(adata_map)
print(adata_map.obs['cortical_layer'].value_counts().reindex(list(scn.params['layers'])).to_dict())
fig = scn.plot_layer_workflow(adata_map, library_id=scn.params['library_main'], df_allen_counts=df_allen_counts,
                              figsize=(16, 7), fontsize_scale=1.6, fname=scn.fname('fig_2I'), 
                              force=force, verbose=verbose)
plt.show()

## 9. Heatmap of relative cell subtype proportion (Fig 2J)

Proportion of each cell type within each layer: Allen = cell counts of the SSp layer dissections; cell2location = mean estimated density per layer, normalised over all mapped types. Shown are the 23 most prevalent SSp-exclusive types (count in SSp > 15, absent from the hippocampal region).

In [ ]:
run_dir_allen = create_dir(scn.file_path('c2l_allen_dir', tag=scn.params['c2l_tag_allen']))

if (run_dir_allen / 'sp.h5ad').exists() or scn.params['c2l_train_if_missing']:
    adata_ctx, df_sig_allen, info_allen = scn.prepare_allen_mapping(adata_allen, adata_map)
    print({k: v for k, v in info_allen.items() if k not in ('types', 'df_type_counts')})
    adata_allen_map = scn.train_mapping(adata_ctx, df_sig_allen, run_dir_allen, N_cells_per_location=scn.params['c2l_allen_N'],
                                       detection_alpha=scn.params['c2l_allen_alpha'], max_epochs=scn.params['c2l_max_epochs'],
                                       batch_key=scn.sample_col, A_factors_per_location=scn.params['c2l_allen_A'], verbose=verbose)
    # layers from the current region labels (the cached obs may predate a change of layer_from_region)
    adata_allen_map.obs['cortical_layer'] = adata_map.obs.loc[adata_allen_map.obs_names, 'cortical_layer'].values
    print(adata_allen_map.shape, {k: v for k, v in adata_allen_map.uns['c2l_training'].items() if k != 'elbo_history'})
else:
    adata_allen_map = None
    print(f"no cached run in {run_dir_allen}: run src/programs/run_c2l_mouse_brain.py --mode allen")

In [ ]:
if adata_allen_map is not None:
    types_top = scn.allen_top_types(df_type_counts)
    df_c2l_prop = scn.calc_df_c2l_layer_prop(adata_allen_map, scn.calc_df_abundance(adata_allen_map).columns)
    fig, stats_2J = scn.plot_layer_heatmaps(df_allen_prop, df_c2l_prop, types_top, figsize=(15, 7), fontsize_scale=1.6, fname=scn.fname('fig_2J'), force=force, verbose=verbose)
    plt.show()
    print(stats_2J)
    df_layer_prop = pd.concat({'allen': df_allen_prop.reindex(types_top), 'cell2location': df_c2l_prop.reindex(types_top)}, axis=1)
    df_layer_prop.columns = [f'{a}_{l}' for a, l in df_layer_prop.columns]
    pdwritecsv(df_layer_prop.round(5).reset_index(names='cell_type'), scn.fname('tsv_layer'), root_result)
    df_layer_prop.round(3)

### Notes for interpretation

- Fig 2C-F come from the paper's own cell2location run; region names are rule-based (**region_celltype_map**, **region_label_override**) and can be edited in **params**.

- Fig 2H and 2J are new cell2location runs. The cached CPU runs use fewer epochs than the paper (30,000), so densities are less converged; check the ELBO curves and rerun on GPU for final figures.
- Fig 2J compares layers defined by **transcriptomic region clusters** across the whole cortex of 2 sections, whereas the paper used histology-based SSp layers of 5 sections. Agreement is therefore expected to be lower and partly reflects how the layer clusters were defined.